# 14 · Melting the chocolate 🍫☕

For the grand finale we melt a bar of chocolate. Press a **warm plate** against its
left edge and watch the **melt front** eat its way in. This is a **Stefan problem** —
a PDE with a *moving boundary* between solid and liquid — and the classic trick is to
stop chasing that boundary explicitly and instead track a smooth **phase field**
$\varphi$ (0 = solid, 1 = molten) that changes rapidly but continuously across a thin
front. It couples to the temperature $T$ through the **latent heat** of melting: turning
solid into liquid *swallows* energy, which holds the front back. Two fields chasing
each other — genuinely **two-way coupled**, and still only a few lines of weak form.

In [ ]:
# --- Google Colab: install NGSolve on first run (a no-op anywhere else) -------
# NGSolve ships its PyPI wheels as pre-releases, so the `--pre` flag is essential.
import sys
if "google.colab" in sys.modules:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "--pre",
                    "ngsolve", "webgui_jupyter_widgets"], check=True)

In [ ]:
from netgen.occ import *
from ngsolve import *
from ngsolve.webgui import Draw

def progress(i, n):                                    # tiny dependency-free bar
    import os
    if os.environ.get("WEBGUI_SCENE_DIR"):             # static-site build: stay silent (no \r spam in the HTML)
        return
    if (i + 1) % max(1, n // 100) == 0 or i + 1 == n:  # (survives JupyterLite/Colab/local)
        b = int(28 * (i + 1) / n)
        sys.stdout.write(f"\r  melting… [{'█'*b}{'·'*(28-b)}] {100*(i+1)//n:3d}%")
        sys.stdout.flush()
        if i + 1 == n:
            sys.stdout.write("\n")

## 1. The bar, and the warm plate

A simple rectangular bar of chocolate. Its left edge is the **`hot`** plate; the rest
of the boundary is left insulated. We keep the mesh modest and use linear elements —
the front is sharp, so resolution near it is what matters, not polynomial degree.

In [ ]:
bar = MoveTo(0, 0).Rectangle(4, 1).Face()
for e in bar.edges: e.name = "outer"
bar.edges.Min(X).name = "hot"
mesh = Mesh(OCCGeometry(bar, dim=2).GenerateMesh(maxh=0.06)); mesh.Curve(2)
print(f"mesh: {mesh.nv} vertices")

## 2. Two coupled fields

**Phase** $\varphi$ and **temperature** $T$ live on the same mesh. The chocolate
*touching* the plate is molten, so $\varphi=1$ there (Dirichlet); the plate itself is
held warm, $T=3$. Everything starts cold and solid: $T=-1$, $\varphi=0$, with the melt
point at $T_m=0$.

In [ ]:
fes  = H1(mesh, order=1, dirichlet="hot")          # phase field
phi, psi = fes.TnT()
gfphi = GridFunction(fes); gfphi.Set(0.0)          # all solid …
gfphi.Set(1.0, definedon=mesh.Boundaries("hot"))   # … except molten at the plate

fesT = H1(mesh, order=1, dirichlet="hot")          # temperature
T, s = fesT.TnT()
gfT = GridFunction(fesT); gfT.Set(-1.0)            # cold solid
gfT.Set(3.0, definedon=mesh.Boundaries("hot"))     # warm plate
Tm = 0.0

## 3. The model — Allen–Cahn phase, heat with latent release

The phase relaxes towards solid or liquid depending on whether the local temperature
is below or above $T_m$, with a thin diffuse interface of width $\sim\!\sqrt{\epsilon^2}$:
$$\tau\,\partial_t\varphi = \epsilon^2\,\Delta\varphi
    \;+\;\varphi(1-\varphi)\big(\varphi-\tfrac12 + m(T)\big),\qquad
    m(T)=\tfrac1\pi\arctan\big(\gamma\,(T-T_m)\big).$$
The double-well factor $\varphi(1-\varphi)$ keeps $\varphi\in[0,1]$; the tilt $m(T)$
decides which well wins. The temperature obeys the heat equation with a **latent**
sink — every bit of melting ($\partial_t\varphi>0$) draws heat out of $T$:
$$\partial_t T = \alpha\,\Delta T \;-\; L\,\partial_t\varphi .$$

In [ ]:
eps2, tau, alphaT, Lheat, gamma = 4e-4, 1e-3, 1.0, 0.6, 4.0
dt, nsteps = 2e-4, 4000

## 4. A variational IMEX scheme

Both equations are **stiff** in their diffusion, so we take that term **implicitly**
and the (local) reaction / coupling **explicitly** — the same IMEX idea as the Turing
system in notebook 15. Each field needs one pre-factorised SPD solve per step
($\tau M+\Delta t\,\epsilon^2 K$ and $M+\Delta t\,\alpha K$), so `sparsecholesky` is
all we need. Inhomogeneous Dirichlet values (the molten/warm plate) are preserved by
updating with the **residual**, never by overwriting the whole vector.

In [ ]:
Mphi = BilinearForm(phi*psi*dx).Assemble()
Kphi = BilinearForm(grad(phi)*grad(psi)*dx).Assemble()
Aphi = Mphi.mat.CreateMatrix()
Aphi.AsVector().data = tau*Mphi.mat.AsVector() + dt*eps2*Kphi.mat.AsVector()
invphi = Aphi.Inverse(fes.FreeDofs(), inverse="sparsecholesky")

MT = BilinearForm(T*s*dx).Assemble()
KT = BilinearForm(grad(T)*grad(s)*dx).Assemble()
mstarT = MT.mat.CreateMatrix()
mstarT.AsVector().data = MT.mat.AsVector() + dt*alphaT*KT.mat.AsVector()
invT = mstarT.Inverse(fesT.FreeDofs(), inverse="sparsecholesky")

# the explicit reaction r(φ, T), tested — reassembled each step from the current state
m = (1/pi)*atan(gamma*(gfT - Tm))
react = LinearForm(gfphi*(1 - gfphi)*(gfphi - 0.5 + m)*psi*dx)

## 5. Let it melt

We store a snapshot of $\varphi$ every so often so the result can be **animated**, and
print a little progress bar — the front advances quickly at first, then slows as it has
to wait for heat to diffuse deeper into the cold bar.

In [ ]:
phi_old = gfphi.vec.CreateVector()
gfphi.AddMultiDimComponent(gfphi.vec)                   # frame 0
with TaskManager():
    for step in range(nsteps):
        phi_old.data = gfphi.vec
        react.Assemble()
        rhs = (tau*Mphi.mat*gfphi.vec + dt*react.vec).Evaluate()
        gfphi.vec.data += invphi*(rhs - Aphi*gfphi.vec).Evaluate()    # implicit φ-diffusion
        dphi = (gfphi.vec - phi_old).Evaluate()
        rhsT = (MT.mat*gfT.vec - Lheat*MT.mat*dphi).Evaluate()
        gfT.vec.data += invT*(rhsT - mstarT*gfT.vec).Evaluate()       # latent-heat sink
        if step % 200 == 0:
            gfphi.AddMultiDimComponent(gfphi.vec)       # store a frame
        progress(step, nsteps)

molten = Integrate(gfphi, mesh)/4.0
print(f"molten fraction: {molten:.0%} of the bar")
Draw(gfphi, mesh, "phase (1 = molten)", interpolate_multidim=True, animate=True,
     min=0, max=1, autoscale=False)

Press play: the melt front sweeps in from the warm plate and then **stalls** — held
up not by the equation's stiffness but by physics, because melting the next layer
first requires conducting heat through everything already molten. That self-limiting
front is the whole point of a Stefan problem, and we never had to track the boundary
by hand.

## Where the ☕ has taken us

Geometry, coefficient functions, spaces, weak forms, solvers, time stepping,
nonlinearity, coupling, PDEs on curved surfaces, and now a moving phase front — the
same handful of ideas, each notebook a little bolder. And this is only the trailhead:
the real **Grand Expedition** — the rest of the **NGSolve User Meeting** and its
fantastic contributions — sets off from
exactly here. Pack your coffee. ☕

![The pirate leads a caravan of riders on rainbow mesh tori toward the mountains —
the Grand Expedition.](data/expedition.jpg)

In [ ]:
# Navigation between units — shown only in a live notebook (Colab / JupyterLite /
# local Jupyter), never in the rendered website (which has its own prev/next nav).
import os, sys
if not os.environ.get("WEBGUI_SCENE_DIR"):          # not the static site build
    _prev = ("13-thermal-plume-hdg", "13 · A puffing thermal plume — HDiv-HDG & HDG 🔥🌀")
    _next = ("15-outlook-unfitted", "15 · Outlook — unfitted FEM with ngsxfem 🫧")
    def _u(_nb):
        if "google.colab" in sys.modules:
            return "https://colab.research.google.com/github/schruste/ngsum2026-colab/blob/colab/" + _nb + ".ipynb"
        return _nb + ".ipynb"                       # JupyterLite & local: relative .ipynb link
    _parts  = ["⬅️ **Previous:** [%s](%s)" % (_prev[1], _u(_prev[0]))] if _prev else []
    _parts += ["➡️ **Next:** [%s](%s)" % (_next[1], _u(_next[0]))] if _next else []
    from IPython.display import display, Markdown
    display(Markdown(" · ".join(_parts)))